TOOLS

Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [3]:
from langchain.chat_models import init_chat_model

model_without_tools = init_chat_model("groq:openai/gpt-oss-20b")
model_without_tools

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.5', 'langchain': '1.4.2'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000002026B3AD160>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002026B3ADE80>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [4]:
from langchain.tools import tool

@tool
def get_weather(city : str) -> str:
    """Get the current weather for a given city."""
    return f"The current weather in {city} is clear and pleasant."

model_with_tools = model_without_tools.bind_tools([get_weather])
model_with_tools

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.5', 'langchain': '1.4.2'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000002026B3AD160>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002026B3ADE80>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'get_weather', 'description': 'Get the current weather for a given city.', 'parameters': {'p

In [5]:
response = model_with_tools.invoke("What is the weather like in New Yourk City?")
response

AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call get_weather function with city "New York City".', 'tool_calls': [{'id': 'fc_0976b129-a03f-4652-b2fc-96da3e4ff5fc', 'function': {'arguments': '{"city":"New York City"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 40, 'prompt_tokens': 133, 'total_tokens': 173, 'completion_time': 0.042951046, 'completion_tokens_details': {'reasoning_tokens': 15}, 'prompt_time': 0.007585559, 'prompt_tokens_details': None, 'queue_time': 0.341987177, 'total_time': 0.050536605}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_587d0d67cc', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0e153-47d6-7763-87e5-7e51d6850f46-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'New York City'}, 'id': 'fc_0976b129-a03f-4652-b2fc-96da3e4ff5fc', 'type': 'tool_call'}], invalid_tool_calls=[], us

In [6]:
for tool_call in response.tool_calls:
    print(f"Tool name : {tool_call["name"]}")
    print(f"Tool arguments : {tool_call["args"]}")

Tool name : get_weather
Tool arguments : {'city': 'New York City'}


TOOL EXECUTION LOOP

In [13]:
messages = [{"role" : "user", "content" : "What is the weather like in New Yourk City?"}]
llm_result = model_with_tools.invoke(messages)
messages.append(llm_result)
for tool_call in llm_result.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)
final_llm_result = model_with_tools.invoke(messages)
messages.append(final_llm_result)
messages

[{'role': 'user', 'content': 'What is the weather like in New Yourk City?'},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "What is the weather like in New Yourk City?" There\'s a typo: "New Yourk City" probably means "New York City". We should correct that. We need to provide the weather. We can use the get_weather function. The function expects "city" string. We should call get_weather with city="New York City". The function returns any. We need to output the result. Let\'s do that.', 'tool_calls': [{'id': 'fc_c95fed1e-5f0a-4277-ba58-eac9472c5708', 'function': {'arguments': '{"city":"New York City"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 115, 'prompt_tokens': 133, 'total_tokens': 248, 'completion_time': 0.12337394, 'completion_tokens_details': {'reasoning_tokens': 90}, 'prompt_time': 0.007570861, 'prompt_tokens_details': None, 'queue_time': 0.309060677, 'total_time': 0.130944801}, 'mode